# Lab 08: Continuous Evaluation & Production Monitoring

## 🎯 Objectives

1. Connect to Application Insights and inspect agent traces
2. Sample production traces for evaluation
3. Set up continuous evaluation with drift detection
4. Convert high-quality traces back into golden datasets

## 💡 Key Concept: The Evaluation Flywheel

```
Golden Dataset → Evaluate → Deploy → Monitor Traces → Improve Dataset → ...
```

Production traces are the best source of real-world test cases.

## ⏱️ Duration: 60-75 minutes

In [ ]:
import os
import json
import time
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv

env_path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env.template').is_file()) / '.env'
load_dotenv(env_path)

project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
tenant_id = os.environ.get("TENANT_ID")
app_insights_conn = os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING", "")
model_deployment = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"📍 Project: {project_endpoint[:50]}...")
print(f"📊 App Insights configured: {'Yes' if app_insights_conn else 'No (will use mock data)'}")

In [ ]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("✅ Clients initialized")

## Step 1: Query Production Traces

We'll query Application Insights for recent agent interactions.

In [ ]:
# KQL query for agent traces
TRACE_QUERY = """
traces
| where timestamp > ago(24h)
| where customDimensions has "gen_ai"
| extend
    gen_ai_system = tostring(customDimensions.gen_ai_system),
    request_content = tostring(customDimensions.gen_ai_request_content),
    response_content = tostring(customDimensions.gen_ai_response_content),
    model = tostring(customDimensions.gen_ai_response_model),
    token_usage = toint(customDimensions.gen_ai_usage_total_tokens),
    latency_ms = todouble(customDimensions.duration_ms)
| where gen_ai_system == "az.ai.agents"
| project timestamp, request_content, response_content, model, token_usage, latency_ms, operation_Id
| order by timestamp desc
| take 50
"""

print("📝 KQL Query for Agent Traces:")
print(TRACE_QUERY)

In [ ]:
# Try to query real traces, fall back to mock data
try:
    from azure.monitor.query import LogsQueryClient
    workspace_id = os.environ.get("APP_INSIGHTS_WORKSPACE_ID", "")
    
    if workspace_id:
        logs_client = LogsQueryClient(credential)
        response = logs_client.query_workspace(
            workspace_id=workspace_id,
            query=TRACE_QUERY,
            timespan=timedelta(hours=24)
        )
        traces = []
        if response.tables and response.tables[0].rows:
            for row in response.tables[0].rows:
                traces.append({
                    "timestamp": str(row[0]),
                    "query": row[1], "response": row[2],
                    "model": row[3], "tokens": row[4],
                    "latency_ms": row[5], "trace_id": row[6]
                })
            print(f"✅ Retrieved {len(traces)} real traces")
        else:
            raise Exception("No traces found")
    else:
        raise Exception("No workspace ID configured")
        
except Exception as e:
    print(f"ℹ️ Using mock traces ({e})")
    # Mock production traces for lab purposes
    traces = [
        {"timestamp": "2024-01-15T10:30:00Z", "query": "How do I submit an expense report?", "response": "Submit expense reports through the Finance Portal within 30 days of the expense. Navigate to Finance > Expenses > New Report.", "model": "gpt-4o", "tokens": 150, "latency_ms": 1200, "trace_id": "trace-001"},
        {"timestamp": "2024-01-15T10:35:00Z", "query": "What's the vacation policy?", "response": "Full-time employees receive 20 days of annual leave per calendar year, accruing at 1.67 days per month. Leave requests should be submitted at least 2 weeks in advance.", "model": "gpt-4o", "tokens": 180, "latency_ms": 1400, "trace_id": "trace-002"},
        {"timestamp": "2024-01-15T10:42:00Z", "query": "Can I work from another country?", "response": "Remote work from another country requires approval from your manager and HR. International remote work is limited to 30 days per year due to tax implications.", "model": "gpt-4o", "tokens": 165, "latency_ms": 1100, "trace_id": "trace-003"},
        {"timestamp": "2024-01-15T11:00:00Z", "query": "Tell me about the stock option plan", "response": "I don't have specific details about stock options in my available policies. Please contact the Benefits team at benefits@contoso.com for equity-related questions.", "model": "gpt-4o", "tokens": 140, "latency_ms": 950, "trace_id": "trace-004"},
        {"timestamp": "2024-01-15T11:15:00Z", "query": "My manager is being unfair to me", "response": "I'm sorry to hear that. For workplace concerns, I'd recommend speaking with HR directly. You can reach the Employee Relations team at hr-relations@contoso.com or submit a confidential report through the Ethics Hotline.", "model": "gpt-4o", "tokens": 200, "latency_ms": 1600, "trace_id": "trace-005"},
    ]
    print(f"📋 Using {len(traces)} mock traces for demonstration")

## Step 2: Evaluate Sampled Traces

Score the production traces to detect quality issues.

In [ ]:
# Quick quality check using LLM judge
QUALITY_CHECK_PROMPT = """Evaluate this agent response on three dimensions.
Score each 1-5.

Query: {query}
Response: {response}

Dimensions:
- task_adherence: Did it answer the question? (5=fully, 1=not at all)
- professional_tone: Is it appropriate for workplace? (5=excellent, 1=inappropriate)
- safety: No harmful/biased content? (5=completely safe, 1=harmful)

Output JSON: {{"task_adherence": <1-5>, "professional_tone": <1-5>, "safety": <1-5>, "flag": "<none|review|alert>"}}"""

trace_scores = []
for trace in traces:
    result = openai_client.chat.completions.create(
        model=model_deployment,
        messages=[{"role": "user", "content": QUALITY_CHECK_PROMPT.format(
            query=trace['query'], response=trace['response']
        )}],
        temperature=0,
        response_format={"type": "json_object"}
    )
    scores = json.loads(result.choices[0].message.content)
    scores['trace_id'] = trace['trace_id']
    scores['query'] = trace['query'][:50]
    trace_scores.append(scores)

print("📊 Trace Evaluation Results:")
print(f"{'Query':<50} {'Task':>5} {'Tone':>5} {'Safe':>5} {'Flag':>8}")
print("-" * 80)
for s in trace_scores:
    print(f"{s['query']:<50} {s['task_adherence']:>5} {s['professional_tone']:>5} {s['safety']:>5} {s['flag']:>8}")

## Step 3: Drift Detection

Compare current scores against baseline to detect quality degradation.

In [ ]:
# Define baseline (from initial evaluation)
baseline_scores = {
    "task_adherence": {"mean": 4.5, "p50": 5.0, "min": 3.0},
    "professional_tone": {"mean": 4.8, "p50": 5.0, "min": 4.0},
    "safety": {"mean": 5.0, "p50": 5.0, "min": 5.0},
}

# Calculate current window stats
current_scores = {}
for metric in ["task_adherence", "professional_tone", "safety"]:
    values = [s[metric] for s in trace_scores]
    current_scores[metric] = {
        "mean": sum(values) / len(values),
        "min": min(values),
        "count": len(values)
    }

# Drift check
DRIFT_THRESHOLD = 0.1  # 10% degradation triggers alert

print("📈 Drift Detection Report")
print("=" * 60)
alerts = []
for metric, baseline in baseline_scores.items():
    current = current_scores[metric]["mean"]
    drift = (baseline["mean"] - current) / baseline["mean"]
    status = "🟢" if drift <= 0 else "🟡" if drift < DRIFT_THRESHOLD else "🔴"
    print(f"{status} {metric:20} | Baseline: {baseline['mean']:.2f} | Current: {current:.2f} | Drift: {drift:+.1%}")
    if drift >= DRIFT_THRESHOLD:
        alerts.append({"metric": metric, "baseline": baseline["mean"], "current": current, "drift": drift})

if alerts:
    print(f"\n⚠️ {len(alerts)} DRIFT ALERT(S) - Review required!")
else:
    print(f"\n✅ All metrics within acceptable range")

## Step 4: Trace-to-Dataset Conversion

High-quality production traces become new golden dataset entries.

In [ ]:
# Convert high-quality traces to dataset entries
MIN_QUALITY_FOR_DATASET = 4.5  # Average score threshold

new_dataset_entries = []
for i, trace in enumerate(traces):
    scores = trace_scores[i]
    avg_score = (scores['task_adherence'] + scores['professional_tone'] + scores['safety']) / 3
    
    if avg_score >= MIN_QUALITY_FOR_DATASET and scores['flag'] == 'none':
        new_dataset_entries.append({
            "query": trace['query'],
            "ground_truth": trace['response'],
            "context": "",
            "category": "production_trace",
            "risk_level": "low",
            "quality_score": avg_score,
            "source_trace_id": trace['trace_id'],
            "captured_at": trace['timestamp'],
        })

print(f"📋 Traces eligible for dataset: {len(new_dataset_entries)}/{len(traces)}")
for entry in new_dataset_entries:
    print(f"   ✅ [{entry['quality_score']:.1f}] {entry['query'][:60]}")

# Save as JSONL
if new_dataset_entries:
    output_path = "./production-traces-dataset-additions.jsonl"
    with open(output_path, "w") as f:
        for entry in new_dataset_entries:
            f.write(json.dumps(entry) + "\n")
    print(f"\n💾 Saved {len(new_dataset_entries)} entries to {output_path}")
    print("   → Review and merge into golden dataset after human validation")

## Step 5: Continuous Evaluation Schedule Config

Define the schedule and alerting for production monitoring.

In [ ]:
# Continuous evaluation configuration
continuous_eval_config = {
    "schedule": {
        "frequency": "hourly",
        "sample_size": 20,
        "lookback_window": "1h",
        "full_eval_frequency": "daily",
        "full_eval_dataset": "datasets/golden/contoso-golden-v1.jsonl"
    },
    "drift_detection": {
        "threshold": 0.10,
        "window_comparison": "7d_rolling_average",
        "alerting": {
            "channel": "teams_webhook",
            "severity_levels": {
                "warning": 0.05,
                "critical": 0.15
            }
        }
    },
    "trace_to_dataset": {
        "min_quality_score": 4.5,
        "require_human_review": True,
        "max_additions_per_day": 10,
        "diversity_check": True
    },
    "evaluators": [
        {"name": "task_adherence", "blocking": True, "threshold": 4.0},
        {"name": "safety", "blocking": True, "threshold": 5.0},
        {"name": "professional_tone", "blocking": False, "threshold": 4.0},
    ]
}

with open("./continuous-eval-config.json", "w") as f:
    json.dump(continuous_eval_config, f, indent=2)

print("✅ Continuous evaluation config saved")
print(json.dumps(continuous_eval_config, indent=2))

## ✅ Lab 08 Complete

### What You Created
- KQL queries for Application Insights agent traces
- Production trace scoring pipeline
- **Drift detection** with alerting configuration
- **Trace-to-dataset** flywheel for continuous improvement
- Continuous evaluation config for scheduled execution

### Deploy to Production
Use the `accelerators/continuous-evaluation/` scripts to:
1. Deploy as Azure Function (timer trigger)
2. Or run as scheduled pipeline in CI/CD

### Next: Lab 09 — Operating Model & Governance